# PlantCV Analysis of MS (aka multispec) sideview (SV) images from experiment PC001 in the Bellwether Phenotyping Facility

Made by: Katie Murphy

Updated: September 15, 2026

We will work through the notebook interactively with one (or a few) test images. After you are happy with the workflow, you will run the analysis over the images in batch (aka parallel). You will not analyze the outputs from this workflow, it is just to set up your script. 

## Step 1: Import Packages

First, import the necessary package. If you get an error here, make sure your kernel (upper right) is set to PlantCV, and that you have PlantCV installed. 

In [ ]:
# Use the interactive matplotlib backend so plots can be zoomed/panned in the notebook
%matplotlib widget

# Import the main PlantCV module (all image processing functions live here)
from plantcv import plantcv as pcv
# WorkflowInputs bundles your image path(s) and output settings into one object
from plantcv.parallel import WorkflowInputs, workflow_inputs, JupyterConfig
import os

## Step 2: Set Up Workflow Inputs

Next, import one image. The image needs to be in the same location that you are working on; for example, if this notebook is on your computer, the image must be on your computer. You cannot reach images from the server if this notebook is on your computer Remember, always keep your raw images separate from your newly processed images! You will need to change the path to your image, and also if you want specific output folders and directories. 

In [ ]:
jupcon = JupyterConfig()
jupcon.verbose = False
jupcon.checkpoint = False
jupcon.input_dir = "../img/" # path relative to your notebook
jupcon.metadata_regex = {"basename": "VIS_TV.*"} # doing it this way we'd just work from the directory level.
# aka, we'd use the directory to read MS from.
summary, full_meta = jupcon.inspect_dataset()
# here we set the number of workers reactively, in practice you'd just pick an appropriate number.
jupcon.cluster_config["n_workers"] = min(5, sum(full_meta["status"] == "Kept"))
summary

In [ ]:
# when you are ready to run in parallel save the notebook, restart the kernel (just to be safe)
# and run to this cell with `jupcon.run()` and `args = workflow_inputs()` uncommmented

#jupcon.run()

In [ ]:
#args = workflow_inputs()

In [ ]:
# @ignore
# Input/output options
args = WorkflowInputs(
    images=["../img/LED-all-G066/MS385_SV_BP0_0_h0_g170_e68000_v500_6468100_0.png"],  # path(s) to your image(s); pointing at one MS image pulls in the rest of its multispectral set
    names="image1",              # variable name(s) to assign to each image, referenced later as args.image
    result="example_results_oneimage_file.csv",  # filename where extracted trait measurements will be saved
    outdir=".",                 # output directory for results/images ("." = current folder)
    writeimg=False,             # set to True if you want PlantCV to save output images to disk
    debug="plot"                # "plot" displays each processing step inline; use "print" to save step images to disk, or None to turn off debug images
    )

## Step 3: Set Debug and Display Parameters

These settings control how PlantCV shows you feedback while you build your workflow interactively, and don't affect the final trait measurements.

In [ ]:
# Set debug to the global parameter, so every PlantCV function displays its output the same way
pcv.params.debug = args.debug
# Change display settings
pcv.params.dpi = 100            # resolution of displayed debug images
pcv.params.text_size = 2       # font size for annotations PlantCV draws on images
pcv.params.text_thickness = 2  # line thickness for annotations PlantCV draws on images

## Step 4: Read In Your Images

Read in the multispectral (MS) image set, plus the separate chlorophyll (CHL) image.

In [ ]:
# Read in your image, which is based on the path you put above. For multispectral (MS) images, you can direct to a folder or to one image, and the other images in that folder will come along. 

ms = pcv.multispec.read_ms(os.path.dirname(args.image1))

# Workaround: MS_data objects from read_ms() don't yet set these attributes,
# but downstream functions like ms.wavelengths and pcv.analyze.spectral_index() expect them.
ms.wavelengths = list(ms.wavelength_dict.keys())
ms.array_type = "datacube"

In [ ]:
chl_img = pcv.multispec.read_ms(os.path.dirname(args.image1), pattern = 'MS455_((SV|TV))_BP650_(\\d+).*')

In [ ]:
gfp_img = pcv.multispec.read_ms(os.path.dirname(args.image1), pattern = 'MS475_((SV|TV))_BP520_(\\d+).*')

## Step 5: Explore the Multispectral Image

Before masking, take a look at what's inside your MS image: how many wavelength bands it has, which wavelengths they are, and what a single band looks like.

In [ ]:
# evaluate the size of the images and how many bands are in the multispectral image (final number)
# shape is (rows, columns, number_of_wavelength_bands)
ms.array_data.shape

In [ ]:
# evaluate which wavelengths are in the multispectral image
# keys are wavelengths (nm) available in this MS image; values are their index/order in ms.array_data
ms.wavelength_dict

In [ ]:
# As an example, let's look at just one wavelength. We need to call one that we know is in there, from the above list. 
# isolate just the 850 nm image from the multispectral image and plot it
ms850_img = ms.select(wavelength=850, ms=False)  # ms=False returns a plain image array instead of another MS_data object

In [ ]:
# @ignore
pcv.plot_image(ms850_img)

## Step 6: Choose a Masking Strategy

Explore colorspaces to find a channel that best separates the plant from the background, then threshold and clean up that channel to build a binary mask.

In [ ]:
# Look at the colorspace - which of these looks the best for masking? Which channel makes the plant look most distinct?
# You can use individual wavelengths, the CHL image, or the pseudo-RGB image to evaluate the colorspace. 
# Here is an example using pseudo-RGB, which is a combination of three wavelengths (R=850, G=680, B=550) that is generated by PlantCV. 
# This is useful for learning, but because not all days of MS imaging have those wavelengths, you cannot make a single mask for all MS analysis. 

colorspace_img = pcv.visualize.colorspaces(rgb_img=ms.pseudo_rgb)

In [ ]:
# Convert the pseudo-RGB image to grayscale using the a channel, which has good separation in plant from everything else 

# gray_img = pcv.rgb2gray_lab(rgb_img=ms.pseudo_rgb, channel='a')

gray_img = pcv.rgb2gray_cmyk(rgb_img=ms.pseudo_rgb, channel='c')



In [ ]:
# Threshold the grayscale image to make a binary (black/white) mask
# object_type='dark' keeps pixels darker than the threshold value (the plant is dark in the 'a' channel here)
#thresh_a = pcv.threshold.binary(gray_img=gray_img, threshold=110, object_type='dark')

thresh_c = pcv.threshold.binary(gray_img=gray_img, threshold=70, object_type='light')

In [ ]:
# Fill in small holes/noise in the binary mask (removes objects smaller than 50 pixels)
c_fill_image = pcv.fill(bin_img=thresh_c, size=50)

In [ ]:
# Now let's try thresholding on the CHL image, which is present every time we take MS images, so will be consistent for our workflow. 

thresh_chl = pcv.threshold.binary(gray_img=chl_img.array_data, threshold=70, object_type='light')

# We are picking up a lot of other reflectance from the metal, we will filter this in a later step. 

In [ ]:
# Fill in small holes/noise in the binary mask (removes objects smaller than 50 pixels)
chl_fill_image = pcv.fill(bin_img=thresh_chl, size=50)

## Step 7: Define Your Region of Interest and Filter the Mask

Draw a box around where you expect your plant to be, then keep only the parts of your mask that fall inside it.

In [ ]:
# Define the region of interest (ROI). This should include your plant, but not you color card or other things. 

# Inputs: 
#   img - RGB or grayscale image to plot the ROI on 
#   x - The x-coordinate of the upper left corner of the rectangle 
#   y - The y-coordinate of the upper left corner of the rectangle 
#   h - The height of the rectangle 
#   w - The width of the rectangle 

roi1 = pcv.roi.rectangle(img=ms.pseudo_rgb, x=1000, y=960, h=2100, w=1900)

In [ ]:
# Make a new filtered mask that only keeps the plant in your ROI and not objects outside of the ROI
# We have set to partial here so that if a leaf extends outside of your ROI it will still be selected. Switch to "cutto" if you have other plants that are getting selected on accident

# Inputs:
#    mask            = the clean mask you made above
#    roi            = the region of interest you specified above
#    roi_type       = 'partial' (default, for partially inside the ROI), 'cutto', or 
#                     'largest' (keep only largest contour)

kept_mask  = pcv.roi.quick_filter(mask=chl_fill_image, roi=roi1, roi_type='partial')


In [ ]:
# we cover the label
kept_mask[3100:3200, 1817:2000] = 0

In [ ]:
# @ignore
pcv.plot_image(kept_mask)

## Step 8: Chlorophyll Analysis

The chlorophyll image uses MS 455 nm and a bandpass filter of 650 nm to isolate just chlorophyll fluorescence. This image has to be imported separately from the MS stack, since it has a bandpass filter. 

In [ ]:
# Use the mask to analyze the CHL image. This will give you a histogram of the pixel values in the CHL image. 
# n_labels=1 means we're treating the whole masked area as one plant/object
chl_analysis = pcv.analyze.grayscale(gray_img=chl_img.array_data, labeled_mask=kept_mask, n_labels=1, bins=100)
chl_img_pseudo = pcv.visualize.pseudocolor(gray_img = chl_img.array_data.squeeze(), mask=kept_mask, cmap='jet', 
                                           min_value=60, max_value=140)

## Step 9: GFP Analysis

The chlorophyll image uses MS 475 nm and a bandpass filter of 520 nm to isolate just GFP fluorescence. This image has to be imported separately from the MS stack, since it has a bandpass filter. 

Note that you will still see some signal even with no GFP due to autofluorescence. It is important to always compare your results to a control plant with no GFP. 

In [ ]:
# Use the mask to analyze the GFP image. This will give you a histogram of the pixel values in the GFP image. 
# n_labels=1 means we're treating the whole masked area as one plant/object

gfp_analysis = pcv.analyze.grayscale(gray_img=gfp_img.array_data, labeled_mask=kept_mask, n_labels=1, bins=100)

gfp_img_pseudo = pcv.visualize.pseudocolor(gray_img = gfp_img.array_data.squeeze(), mask=kept_mask, cmap='jet', 
                                           min_value=25, max_value=60)

## Step 10: Spectral Index Analysis

Calculate vegetation/pigment indices from the MS image bands, then analyze and visualize each one. Indices used here:
- **NDVI** (Normalized Difference Vegetation Index) - general plant vigor/greenness
- **SR** (Simple Ratio) - another general vegetation index, ratio-based
- **PSND_chlb** (Pigment Specific Normalized Difference, chlorophyll b) - chlorophyll b content
- **PSSR_chlb** (Pigment Specific Simple Ratio, chlorophyll b) - chlorophyll b content, ratio-based
- **SAVI** (Soil Adjusted Vegetation Index) - vegetation index that reduces soil brightness influence
- **EVI** (Enhanced Vegetation Index) - vegetation index correcting for atmospheric/canopy background effects
- **GLI** (Green Leaf Index) - visible-bands-only vegetation index computed from the pseudo-RGB image
- **VARI** (Visible Atmospherically Resistant Index) - vegetation index using only visible wavelengths
- **VI_GREEN** (Vegetation Index Green) - vegetation index using the green and red bands
- **GDVI** (Green Difference Vegetation Index) - vegetation index using the difference between NIR and green bands
- **EGI** (Excess Green Index) - visible-bands-only vegetation index computed from the pseudo-RGB image
- **ARI** (Anthocyanin Reflectance Index) - anthocyanin content
- **MARI** (Modified Anthocyanin Reflectance Index) - anthocyanin content, adjusted for chlorophyll/NIR reflectance
- **CRI550** (Carotenoid Reflectance Index 550) - carotenoid content
- **CRI700** (Carotenoid Reflectance Index 700) - carotenoid content
- **SIPI** (Structure-Independent Pigment Index) - carotenoid-to-chlorophyll ratio
- **PSND_CHLA** (Pigment Specific Normalized Difference, chlorophyll a) - chlorophyll a content
- **PSSR_CHLA** (Pigment Specific Simple Ratio, chlorophyll a) - chlorophyll a content, ratio-based
- **PSND_CAR** (Pigment Specific Normalized Difference, carotenoids) - carotenoid content
- **PSSR_CAR** (Pigment Specific Simple Ratio, carotenoids) - carotenoid content, ratio-based
- **PSRI** (Plant Senescence Reflectance Index) - senescence/stress indicator
- **CI_REDEDGE** (Chlorophyll Index Red Edge) - chlorophyll content
- **NDCI** (Normalized Difference Chlorophyll Index) - chlorophyll content
- **RGRI** (Red/Green Ratio Index) - anthocyanin-to-chlorophyll ratio

In [ ]:
# NDVI (Normalized Difference Vegetation Index) - a general measure of plant vigor/greenness
# Calculate the index for the whole image
ndvi_index = pcv.spectral_index.ndvi(ms, distance=55)
# Workaround: index images built by pcv.spectral_index functions don't set array_type themselves,
# but pcv.analyze.spectral_index() needs it - see notebook history for the underlying PlantCV bug
ndvi_index.array_type = "index_ndvi"

In [ ]:
# Graph the histogram of the spectral index, masking so that you only have the plant
# min_bin/max_bin set the expected value range for the histogram bins; widen these if you see a range warning
ndvi_output=pcv.analyze.spectral_index(index_img=ndvi_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)

In [ ]:
# Pseudocolor the image as a visualization - maps NDVI values to a color scale so you can see spatial patterns
ndvi_img = pcv.visualize.pseudocolor(gray_img = ndvi_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# SR (Simple Ratio) - another general vegetation vigor index
# Calculate the index for the whole image
sr_index = pcv.spectral_index.sr(ms, distance=55)
sr_index.array_type = "index_sr"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
sr_output=pcv.analyze.spectral_index(index_img=sr_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=5)
# Pseudocolor the image as a visualization
sr_img = pcv.visualize.pseudocolor(gray_img = sr_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=5)

In [ ]:
# psnd_chlb (Pigment Specific Normalized Difference, chlorophyll b) - relates to chlorophyll b content
# Calculate the index for the whole image
psnd_chlb_index = pcv.spectral_index.psnd_chlb(ms, distance=55)
psnd_chlb_index.array_type = "index_psnd_chlb"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
psnd_chlb_output=pcv.analyze.spectral_index(index_img=psnd_chlb_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
psnd_chlb_img = pcv.visualize.pseudocolor(gray_img = psnd_chlb_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# pssr_chlb (Pigment Specific Simple Ratio, chlorophyll b) - ratio-based chlorophyll b content index
# Calculate the index for the whole image
pssr_chlb_index = pcv.spectral_index.pssr_chlb(ms, distance=55)
pssr_chlb_index.array_type = "index_pssr_chlb"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
pssr_chlb_output=pcv.analyze.spectral_index(index_img=pssr_chlb_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=5)
# Pseudocolor the image as a visualization
pssr_chlb_img = pcv.visualize.pseudocolor(gray_img = pssr_chlb_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=5)

In [ ]:
# SAVI (Soil Adjusted Vegetation Index) - like NDVI, but reduces the influence of soil brightness in the background
# Calculate the index for the whole image
savi_index = pcv.spectral_index.savi(ms, distance=55)
savi_index.array_type = "index_savi"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
savi_output=pcv.analyze.spectral_index(index_img=savi_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
savi_img = pcv.visualize.pseudocolor(gray_img = savi_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# EVI (Enhanced Vegetation Index) - vegetation index that corrects for atmospheric and canopy background effects
# Calculate the index for the whole image
evi_index = pcv.spectral_index.evi(ms, distance=55)
evi_index.array_type = "index_evi"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
evi_output=pcv.analyze.spectral_index(index_img=evi_index, labeled_mask=kept_mask,
                           min_bin=-2, max_bin=1)
# Pseudocolor the image as a visualization
evi_img = pcv.visualize.pseudocolor(gray_img = evi_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-2, max_value=1)

In [ ]:
# GLI (Green Leaf Index)
# Calculate the index for the whole image
gli_index = pcv.spectral_index.gli(ms)
gli_index.array_type = "index_gli"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
gli_output=pcv.analyze.spectral_index(index_img=gli_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
gli_img = pcv.visualize.pseudocolor(gray_img = gli_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# VARI (Visible Atmospherically Resistant Index) - vegetation index using only visible wavelengths
# Calculate the index for the whole image
vari_index = pcv.spectral_index.vari(ms, distance=55)
vari_index.array_type = "index_vari"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
vari_output=pcv.analyze.spectral_index(index_img=vari_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
vari_img = pcv.visualize.pseudocolor(gray_img = vari_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# VI_GREEN (Vegetation Index - Green) - vegetation index using the green and red bands
# Calculate the index for the whole image
vi_green_index = pcv.spectral_index.vi_green(ms, distance=55)
vi_green_index.array_type = "index_vi_green"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
vi_green_output=pcv.analyze.spectral_index(index_img=vi_green_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
vi_green_img = pcv.visualize.pseudocolor(gray_img = vi_green_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# GDVI (Green Difference Vegetation Index) - vegetation index based on the difference between NIR and green bands
# Calculate the index for the whole image
gdvi_index = pcv.spectral_index.gdvi(ms, distance=55)
gdvi_index.array_type = "index_gdvi"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
gdvi_output=pcv.analyze.spectral_index(index_img=gdvi_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
gdvi_img = pcv.visualize.pseudocolor(gray_img = gdvi_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# EGI (Excess Green Index) 
# Calculate the index for the whole image
egi_index = pcv.spectral_index.egi(ms, distance = 50)
egi_index.array_type = "index_egi"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
egi_output=pcv.analyze.spectral_index(index_img=egi_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
egi_img = pcv.visualize.pseudocolor(gray_img = egi_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# ARI - Anthocyanin Reflectance Index
# Calculate the index for the whole image
ari_index = pcv.spectral_index.ari(ms, distance=55)
ari_index.array_type = "index_ari"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
ari_output=pcv.analyze.spectral_index(index_img=ari_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
ari_img = pcv.visualize.pseudocolor(gray_img = ari_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.2, max_value=0.2)

In [ ]:
# MARI - Modified Anthocyanin Reflectance Index
# Calculate the index for the whole image
mari_index = pcv.spectral_index.mari(ms, distance=55)
mari_index.array_type = "index_mari"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
mari_output=pcv.analyze.spectral_index(index_img=mari_index, labeled_mask=kept_mask,
                           min_bin=-2, max_bin=1)
# Pseudocolor the image as a visualization
mari_img = pcv.visualize.pseudocolor(gray_img = mari_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-2, max_value=1)

In [ ]:
# CRI550 - Carotenoid Reflectance Index 550
# Calculate the index for the whole image
cri550_index = pcv.spectral_index.cri550(ms, distance=55)
cri550_index.array_type = "index_cri550"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
cri550_output=pcv.analyze.spectral_index(index_img=cri550_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
cri550_img = pcv.visualize.pseudocolor(gray_img = cri550_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# CRI700 - Carotenoid Reflectance Index 700
# Calculate the index for the whole image
cri700_index = pcv.spectral_index.cri700(ms, distance=55)
cri700_index.array_type = "index_cri700"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
cri700_output=pcv.analyze.spectral_index(index_img=cri700_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
cri700_img = pcv.visualize.pseudocolor(gray_img = cri700_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# SIPI - Structure-Independent Pigment Index
# Calculate the index for the whole image
sipi_index = pcv.spectral_index.sipi(ms, distance=55)
sipi_index.array_type = "index_sipi"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
sipi_output=pcv.analyze.spectral_index(index_img=sipi_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=2)
# Pseudocolor the image as a visualization
sipi_img = pcv.visualize.pseudocolor(gray_img = sipi_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=2)

In [ ]:
# PSND_CHLA - Pigment Specific Normalized Difference, chlorophyll a
# Calculate the index for the whole image
psnd_chla_index = pcv.spectral_index.psnd_chla(ms, distance=55)
psnd_chla_index.array_type = "index_psnd_chla"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
psnd_chla_output=pcv.analyze.spectral_index(index_img=psnd_chla_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
psnd_chla_img = pcv.visualize.pseudocolor(gray_img = psnd_chla_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# PSSR_CHLA - Pigment Specific Simple Ratio, chlorophyll a
# Calculate the index for the whole image
pssr_chla_index = pcv.spectral_index.pssr_chla(ms, distance=55)
pssr_chla_index.array_type = "index_pssr_chla"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
pssr_chla_output=pcv.analyze.spectral_index(index_img=pssr_chla_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=2)
# Pseudocolor the image as a visualization
pssr_chla_img = pcv.visualize.pseudocolor(gray_img = pssr_chla_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=2)

In [ ]:
# PSND_CAR - Pigment Specific Normalized Difference, carotenoids
# Calculate the index for the whole image
psnd_car_index = pcv.spectral_index.psnd_car(ms, distance=55)
psnd_car_index.array_type = "index_psnd_car"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
psnd_car_output=pcv.analyze.spectral_index(index_img=psnd_car_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
psnd_car_img = pcv.visualize.pseudocolor(gray_img = psnd_car_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# PSSR_CAR - Pigment Specific Simple Ratio, carotenoids
# Calculate the index for the whole image
pssr_car_index = pcv.spectral_index.pssr_car(ms, distance=55)
pssr_car_index.array_type = "index_pssr_car"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
pssr_car_output=pcv.analyze.spectral_index(index_img=pssr_car_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
pssr_car_img = pcv.visualize.pseudocolor(gray_img = pssr_car_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# PSRI - Plant Senescence Reflectance Index
# Calculate the index for the whole image
psri_index = pcv.spectral_index.psri(ms, distance=55)
psri_index.array_type = "index_psri"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
psri_output=pcv.analyze.spectral_index(index_img=psri_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
psri_img = pcv.visualize.pseudocolor(gray_img = psri_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# CI_REDEDGE - Chlorophyll Index Red Edge
# Calculate the index for the whole image
ci_rededge_index = pcv.spectral_index.ci_rededge(ms, distance=55)
ci_rededge_index.array_type = "index_ci_rededge"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
ci_rededge_output=pcv.analyze.spectral_index(index_img=ci_rededge_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
ci_rededge_img = pcv.visualize.pseudocolor(gray_img = ci_rededge_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# NDCI - Normalized Difference Chlorophyll Index
# Calculate the index for the whole image
ndci_index = pcv.spectral_index.ndci(ms, distance=55)
ndci_index.array_type = "index_ndci"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
ndci_output=pcv.analyze.spectral_index(index_img=ndci_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
ndci_img = pcv.visualize.pseudocolor(gray_img = ndci_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

In [ ]:
# RGRI - Red/Green Ratio Index
# Calculate the index for the whole image
rgri_index = pcv.spectral_index.rgri(ms, distance=55)
rgri_index.array_type = "index_rgri"  # workaround: set manually since spectral_index functions don't set this
# Graph the histogram of the spectral index, masking so that you only have the plant
rgri_output=pcv.analyze.spectral_index(index_img=rgri_index, labeled_mask=kept_mask,
                           min_bin=-1, max_bin=1)
# Pseudocolor the image as a visualization
rgri_img = pcv.visualize.pseudocolor(gray_img = rgri_index.array_data, mask=kept_mask, cmap='jet', 
                                           min_value=-0.25, max_value=1)

## Step 11: Shape Analysis

You can analyze shape either with the multispec image(s), or the RGB images. We will show analyze shape here to be thorough, but you do not need to do it on both image types, and we recommend using RGB instead because it will probably have more angles. 

In [ ]:
############### Shape Analysis ################ 
  
# Find shape properties, data gets stored to an Outputs class automatically

# Inputs:
#   img - RGB or grayscale image data 
#   labeled_mask - the mask of each individual object, set by the create_labels function. 
#   n_labels - the number of objects, set by the create_labels function. 

shape_image = pcv.analyze.size(img=ms.pseudo_rgb, labeled_mask=kept_mask)


In [ ]:
# Shape properties relative to user boundary line (optional). This is useful if your plant is hanging below the pot and you want height from the top of the pot.
# Set your line_position by finding the x-value at the top of the pot, hover your cursor to get that value

# Inputs: 
#   img - RGB or grayscale image data 
#   obj - Single or grouped contour object 
#   mask - Binary mask of selected contours 
#   line_position - Position of boundary line (a value of 0 would draw a line 
#                   through the bottom of the image) 
#   label - Optional label parameter, modifies the variable name of observations recorded. (default `label="default"`)filled_img = pcv.morphology.fill_segments(mask=cropped_mask, objects=edge_objects)

shape_boundary_image = pcv.analyze.bound_horizontal(img=ms.pseudo_rgb, labeled_mask=kept_mask, 
                                               line_position=3075)


## Step 12: Save Your Results

In [ ]:
# The save results function will take the measurements stored when running any PlantCV analysis functions, format, 
# and print an output text file for data analysis. The Outputs class stores data whenever any of the following functions
# are ran: analyze_bound_horizontal, analyze_bound_vertical, analyze_color, analyze_nir_intensity, analyze_object, 
# fluor_fvfm, report_size_marker_area, watershed. If no functions have been run, it will print an empty text file 

#This saves results for one image, and each image is saved individually if you run another image (it will overwrite the last one)
pcv.outputs.save_results(filename=args.result)


## Step 13: Build Your Batch Workflow

Congrats, you now know all the settings you like for this day of imaging! It's time to make this into a workflow so that it will analyze all your images at once and you can go have a cup of coffee. To do so, go back to the folder and open up the config_template.json file, and the config_workflow.py files, and you will edit them according to the values you changed in this file. 